# Модерация токсичных комментариев (HF Datasets + Polars + Transformers)

В этом ноутбуке строим простой и понятный пайплайн классификации токсичности текста на датасете Civil Comments.

- Используем Polars для анализа и манипуляций с табличными данными.
- Обучим DistilBERT при помощи Hugging Face Transformers (Trainer).
- После каждого крупного шага дадим короткую проверку результата (1–2 предложения).


In [2]:
import polars as pl
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import transformers
from sklearn.metrics import f1_score, precision_score, recall_score

In [3]:
# Фиксируем seed для воспроизводимости
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("polars:", pl.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

polars: 1.33.1
transformers: 4.56.1
torch: 2.8.0+cu128 CUDA: True


### Выбор датасета 
**https://huggingface.co/datasets/google/civil_comments**
Датасет с пользовательскими комментариями и разметкой токсичности (общая токсичность и аспекты).

Поля и их смысл (ядро): 

- text (string) — текст комментария (в датасете поля называются "text" ).
- toxicity (float) — уровень токсичности в [0,1] (continuous). Также присутствуют поля severe_toxicity, identity_attack, insult, threat, obscene и др. (как вероятности/оценки).


### Задача:
обучить модель на датасете  из HF  и испоользовать для классификациии комментиариев 

In [4]:
# Пример  загрузки с HF Hub 

splits = {
    'train': 'data/train-*.parquet',
    'validation': 'data/validation-00000-of-00001.parquet',
    'test': 'data/test-00000-of-00001.parquet'
}

# Полная версия всех сплитов:
# df_train = pl.read_parquet('hf://datasets/google/civil_comments/' + splits['train'])
# df_val = pl.read_parquet('hf://datasets/google/civil_comments/' + splits['validation'])
# df_test = pl.read_parquet('hf://datasets/google/civil_comments/' + splits['test'])
# df_train.write_parquet("../data/civil_comments_train.parquet")
# df_val.write_parquet("../data/civil_comments_val.parquet")
# df_test.write_parquet("../data/civil_comments_test.parquet")

In [5]:
# Загрузка данных из локальных Parquet (уже сохранены заранее)
df_train = pl.read_parquet('../data/civil_comments_train.parquet')
df_val = pl.read_parquet('../data/civil_comments_val.parquet')
df_test = pl.read_parquet('../data/civil_comments_test.parquet')


In [6]:
df_train.sample(5)

text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
str,f32,f32,f32,f32,f32,f32,f32
"""Governor Walker and Alaska's c…",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Wow great analogy !!! . Most …",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Once again you are correct D G…",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Bob, Did I say anything about…",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""How are they second class citi…",0.0,0.0,0.0,0.0,0.0,0.0,0.0


виды комментариев: 
- toxicity — float32. Оценка   токсичности комментария.
- severe_toxicity — float32. Оценка  “сильно токсичного” контента (более жёсткая/агрессивная токсичность).
- obscene — float32. Оценка  наличия обсценной лексики (брань/непристойности).
- threat — float32. Оценка  наличия угроз (намёков/призывов к насилию и т.п.).
- insult — float32. Оценка  наличия оскорблений.
- identity_attack — float32. Оценка  нападок по признаку идентичности (раса, гендер, религия и т.д.).
- sexual_explicit — float32. Оценка  сексуально откровенного содержания.

In [7]:
df_train.select(
    ((pl.col("toxicity") > 0).sum() / pl.len()).alias("toxicity > 0 share"), 
    ((pl.col("severe_toxicity") > 0).sum() / pl.len()).alias("severe_toxicity > 0 share"),
    ((pl.col("obscene") > 0).sum() / pl.len()).alias("obscene > 0 share"),
    ((pl.col("threat") > 0).sum() / pl.len()).alias("threat > 0 share"),
    ((pl.col("insult") > 0).sum() / pl.len()).alias("insult > 0 share"),
    ((pl.col("identity_attack") > 0).sum() / pl.len()).alias("identity_attack > 0 share"),
    ((pl.col("sexual_explicit") > 0).sum() / pl.len()).alias("sexual_explicit > 0 share")
)

toxicity > 0 share,severe_toxicity > 0 share,obscene > 0 share,threat > 0 share,insult > 0 share,identity_attack > 0 share,sexual_explicit > 0 share
f64,f64,f64,f64,f64,f64,f64
0.299251,0.057974,0.081543,0.059245,0.25184,0.12109,0.038189


видим что есть большой дисбаланс особенено threat, identity_attack, threat, sexual_explicit, поэтому будем предсказывать общий класс токсичности комментария 

In [52]:
pl.Config.set_fmt_str_lengths(300)
df_train.filter(
    (pl.col("toxicity") == 0)
    & ((pl.col("severe_toxicity") > 0)
    | (pl.col("obscene") > 0)
    | (pl.col("threat") > 0)
    | (pl.col("insult") > 0)
    | (pl.col('identity_attack')>0)
    | (pl.col('sexual_explicit')>0))

).sample(3)

text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
str,f32,f32,f32,f32,f32,f32,f32
"""Nelson especially made key fumbles and then compounded the negatives by getting several ""anger management problem"" flags--- Helf showed his wimpiness by not benching Sir Charles for any of them. Then, they were repeated. Nelson baffles me: he's obviously one of the most, if not THE most talented ath…",0.0,0.0,0.166667,0.0,0.0,0.0,0.0
"""I don't see a post from ""Smarty Alec"" today, but perhaps I could answer for him. No one minds admitting the supremacy of God. The supremacy of those who claim to speak for God is much more questionable, which is why we separate church from state.""",0.0,0.0,0.0,0.0,0.0,0.2,0.0
"""Please note I blamed NO victims Bouldergeist. We, the US, meddle constantly in other countries elections. So who are we to cast the first stone? And this Russian meddling falls squarely on Obama primarily, and backwards from there. Not on Trump. Again, not our President Trump, try as the Far L…",0.0,0.0,0.0,0.0,0.166667,0.0,0.0


получается что есть строки которые не попадают в общий уровень токсичности но при этом попадают в жругие параметры по токсичности

In [53]:
df_train.filter(pl.col('insult')> pl.col('toxicity'))

text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
str,f32,f32,f32,f32,f32,f32,f32
"""""Creativity in the classroom is hard to come by. Stay out of its way, please."" Agreed.""",0.0,0.0,0.0,0.0,0.166667,0.0,0.0
"""The Oregonian is hardly what I'd call journalism these days. They've been feeding the far-right propaganda for weeks now. The ignorance is mind-blowing.""",0.0,0.0,0.0,0.0,0.166667,0.0,0.0
"""I just don't really think more planning experts are what we need in government. We need someone who can fix Portland so we can have good, middle class jobs. Planners have a different focus. Quality of life starts with a decent job, not a pedestrian friendly neighborhood where you can walk to your…",0.0,0.0,0.0,0.0,0.2,0.0,0.0
"""The author says that there are > many, many more reasons I am assuming a sequel is in the works, then?""",0.0,0.0,0.166667,0.166667,0.166667,0.166667,0.166667
"""Although the food was pretty good, I had a strange experience at NLB. As a vegetarian who dislikes mushrooms, I always ask restaurant staff whether their veggie burgers are made with mushrooms. The order taker looked like I was crazy. She then asked the kitchen staff. The best they could answer …",0.0,0.0,0.0,0.0,0.2,0.0,0.0
…,…,…,…,…,…,…,…
"""Why respond to him? He's obviously a racist. And a stupid one at that.""",0.857143,0.071429,0.125,0.0,0.875,0.125,0.0
"""Edison sits on the edge of the uber-liberal UO campus. This is where the kids of UO staff attend school. A simple and effective solution is to kick these brats out of public schools and make their parents home school them. Why do liberals tolerate that baloney?""",0.2,0.0,0.0,0.0,0.3,0.0,0.0
"""Too bad he thinks with his little head.""",0.2,0.0,0.0,0.0,0.3,0.0,0.0


In [54]:
df_train.limit(5)

text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
str,f32,f32,f32,f32,f32,f32,f32
"""This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyone get in your way!""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""This is such an urgent design problem; kudos to you for taking it on. Very impressive!""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Is this something I'll be able to install on my site? When will you be releasing it?""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""haha you guys are a bunch of losers.""",0.893617,0.021277,0.0,0.0,0.87234,0.021277,0.0


In [55]:
df_train.columns

['text',
 'toxicity',
 'severe_toxicity',
 'obscene',
 'threat',
 'insult',
 'identity_attack',
 'sexual_explicit']

In [56]:
# Бинаризуем метку общей токсичности (toxic vs non-toxic)
# (в исходном датасете значения — вероятности; по порогу формируем 0/1)
THRESHOLD = 0.5

df_train = (
    df_train
    .select(["text", "toxicity"])  # оставим только нужные колонки
    .with_columns((pl.col("toxicity") >= THRESHOLD).cast(pl.Int8).alias("label"))
    .select(["text", "label"])     # для обучения храним только текст и бинарную метку
)

df_val = (
    df_val
    .select(["text", "toxicity"]) 
    .with_columns((pl.col("toxicity") >= THRESHOLD).cast(pl.Int8).alias("label"))
    .select(["text", "label"]) 
)

df_test = (
    df_test
    .select(["text", "toxicity"]) 
    .with_columns((pl.col("toxicity") >= THRESHOLD).cast(pl.Int8).alias("label"))
    .select(["text", "label"]) 
)

In [57]:

df_train.sample(3)

text,label
str,i8
"""Hope WT's doing some recruiting while he's out there. Come home with a certain Bama coach, if nothing else.""",0
"""""delighted in the killing of the unborn"" delighted?? That is a slur: uncalled for, unnecessary, and untrue.""",0
"""And not without many black voters backing him.""",1


In [58]:

print(df_train.shape, df_val.shape, df_test.shape)


(1804874, 2) (97320, 2) (97320, 2)


In [59]:
# Пример строк
pl.Config.set_fmt_str_lengths(200)
df_train.head(3)

text,label
str,i8
"""This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!""",0
"""Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyone get in your way!""",0
"""This is such an urgent design problem; kudos to you for taking it on. Very impressive!""",0


In [60]:
# Подготовка данных, токенизатор и датасеты (on-the-fly токенизация)
from torch.utils.data import Dataset

# Списки текстов и метки
train_texts = df_train.get_column('text').to_list()
val_texts = df_val.get_column('text').to_list()
test_texts = df_test.get_column('text').to_list()

train_labels = df_train.get_column('label').to_numpy().astype(np.int64)
val_labels = df_val.get_column('label').to_numpy().astype(np.int64)
test_labels = df_test.get_column('label').to_numpy().astype(np.int64)

print('train/val/test sizes:', len(train_texts), len(val_texts), len(test_texts))
print('label shapes:', train_labels.shape, val_labels.shape, test_labels.shape)

# Токенизатор и collator
checkpoint = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
max_length = 256
collator = DataCollatorWithPadding(tokenizer=tokenizer)

class HFTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = HFTextDataset(train_texts, train_labels, tokenizer, max_length=max_length)
val_ds = HFTextDataset(val_texts, val_labels, tokenizer, max_length=max_length)
test_ds = HFTextDataset(test_texts, test_labels, tokenizer, max_length=max_length)


train/val/test sizes: 1804874 97320 97320
label shapes: (1804874,) (97320,) (97320,)


In [61]:

# Быстрый sanity-check одного элемента
sample = train_ds[0]
print({k: tuple(v.shape) if hasattr(v, 'shape') else type(v) for k, v in sample.items()})

{'input_ids': (31,), 'attention_mask': (31,), 'labels': ()}


In [62]:
# Баланс метки (доля токсичных)
pos_rate = float(df_train.select(pl.col('label').mean()).item())
pl.DataFrame({'label': ['toxic'], 'pos_rate': [pos_rate]})

label,pos_rate
str,f64
"""toxic""",0.079969


In [63]:
# Пример строк из обучающего набора 
pl.Config.set_fmt_str_lengths(300)
df_train.select(["text", "label"]).sample(3)

text,label
str,i8
"""Coming from Haiti they better be given an AIDS test before anything else.""",0
"""Yep. No system is perfect, and this one has flaws that probably allow a motivated, agenda-driven person or group to get acceptable comments removed that they don't agree with. I noticed yesterday, while going through the posting process, an earlier comment of mine appeared for review. A few second…",0
"""Bear in mind our oil only has value if we can export it to countries other than the US, if we cut the US off. The US has done a remarkably good job through the Tides Foundation of courting Canadian shills to oppose all access to tidewater for our oil. We are shooting ourselves in the foot if we cu…",0


In [64]:
# Краткая сводка по обучению: количество токсичных комментариев
num_pos = int(df_train.select(pl.col('label').sum()).item())
pl.DataFrame({'label': ['toxic'], 'num_pos': [num_pos]})

label,num_pos
str,i64
"""toxic""",144334


In [65]:
# Модель и Trainer для бинарной классификации
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
)

USE_GPU = torch.cuda.is_available()
print("CUDA:", USE_GPU, (torch.cuda.get_device_name(0) if USE_GPU else 'CPU'))

# Создаём TrainingArguments с учётом версии Transformers:
# некоторые поля (evaluation_strategy/save_strategy/metric_for_best_model/...) отсутствуют в старых версиях
from dataclasses import fields as dataclass_fields

# Базовые (широко поддерживаемые) аргументы
base_kwargs = dict(
    output_dir="./civil_comments_distilbert_bin",  # каталог под бекапы чекпойнтов
    learning_rate=2e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1000,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    seed=SEED,
)

# Проверяем, какие аргументы поддерживаются текущей версией transformers
supported_arg_names = {f.name for f in dataclass_fields(TrainingArguments)}

has_eval = "evaluation_strategy" in supported_arg_names
has_save = "save_strategy" in supported_arg_names
has_best = "load_best_model_at_end" in supported_arg_names
has_metric_for_best = "metric_for_best_model" in supported_arg_names
has_greater = "greater_is_better" in supported_arg_names
has_save_total = "save_total_limit" in supported_arg_names

# Если доступны обе стратегии (eval и save), выставим их синхронно и включим best-модель
if has_eval and has_save:
    base_kwargs["evaluation_strategy"] = "epoch"
    base_kwargs["save_strategy"] = "epoch"
    if has_best:
        base_kwargs["load_best_model_at_end"] = True
    if has_metric_for_best:
        base_kwargs["metric_for_best_model"] = "f1"
    if has_greater:
        base_kwargs["greater_is_better"] = True
    if has_save_total:
        base_kwargs["save_total_limit"] = 2
else:
    # Если нет evaluation_strategy, чтобы избежать конфликтов, отключим best и сохранялку приведём к 'no'
    if has_save:
        base_kwargs["save_strategy"] = "no"
    if has_best:
        base_kwargs["load_best_model_at_end"] = False

args = TrainingArguments(**base_kwargs)



def safe_unpack_eval(eval_pred):
    # Поддержка старого и нового API EvalPrediction
    if isinstance(eval_pred, tuple):
        return eval_pred
    return eval_pred.predictions, eval_pred.label_ids

# Метрики: F1/Precision/Recall по бинарной задаче
def compute_metrics(eval_pred):
    logits, labels = safe_unpack_eval(eval_pred)
    preds = logits.argmax(axis=1)
    return {
        'f1': float(f1_score(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer device:", next(trainer.model.parameters()).device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CUDA: True NVIDIA GeForce RTX 5090
Trainer device: cuda:0


In [69]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [72]:
# Сравнение нескольких чекпоинтов: короткий прогон и выбор лучшего по F1 на валидации
CANDIDATE_CHECKPOINTS = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base",
]
QUICK_MAX_STEPS = 500  # уменьшите/увеличьте при необходимости

from dataclasses import fields as dataclass_fields
from sklearn.metrics import f1_score


def quick_f1(ckpt: str, max_steps: int = QUICK_MAX_STEPS) -> float:
    """Коротко дообучает ckpt и возвращает F1 на валидации (совместимо с разными версиями transformers)."""
    tok = AutoTokenizer.from_pretrained(ckpt)
    train_local = HFTextDataset(train_texts, train_labels, tok, max_length=max_length)
    val_local = HFTextDataset(val_texts, val_labels, tok, max_length=max_length)

    mdl = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)

    # Совместимость: выставляем только поддерживаемые TrainingArguments
    supported = {f.name for f in dataclass_fields(TrainingArguments)}

    use_bf16 = (
        torch.cuda.is_available()
        and hasattr(torch.cuda, "is_bf16_supported")
        and torch.cuda.is_bf16_supported()
        and ("bf16" in supported)
    )

    args_kwargs = dict(
        output_dir=f"./tmp_cmp_{ckpt.replace('/', '_')}",
        learning_rate=2e-5,
        num_train_epochs=1,
        weight_decay=0.01,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        logging_steps=200,
        seed=SEED,
    )
    # Ограничение по шагам, если поле поддерживается
    if "max_steps" in supported:
        args_kwargs["max_steps"] = max_steps
    # Ускорение загрузки батчей, если поддерживается
    if "dataloader_num_workers" in supported:
        args_kwargs["dataloader_num_workers"] = 2
    # bf16 — только если поле есть и GPU поддерживает
    if use_bf16:
        args_kwargs["bf16"] = True

    args_local = TrainingArguments(**args_kwargs)
    coll_local = DataCollatorWithPadding(tokenizer=tok)

    tr = Trainer(
        model=mdl,
        args=args_local,
        train_dataset=train_local,
        eval_dataset=val_local,
        data_collator=coll_local,
        compute_metrics=compute_metrics,
    )

    tr.train()
    metrics = tr.evaluate(eval_dataset=val_local)
    f1 = metrics.get("eval_f1")
    if f1 is None:
        out = tr.predict(val_local)
        preds = out.predictions.argmax(axis=1)
        f1 = float(f1_score(val_labels, preds))
    return float(f1)

# Запуск сравнения
results = []
for ckpt in CANDIDATE_CHECKPOINTS:
    f1 = quick_f1(ckpt)
    print(f"{ckpt}: F1={f1:.4f}")
    results.append((ckpt, f1))

best_ckpt, best_f1 = max(results, key=lambda x: x[1])
print(f"\nBest checkpoint: {best_ckpt} (F1={best_f1:.4f})")

# Пересоберём глобальные объекты под лучший чекпоинт
checkpoint = best_ckpt

# Токенизатор/датасеты/коллатор
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
train_ds = HFTextDataset(train_texts, train_labels, tokenizer, max_length=max_length)
val_ds = HFTextDataset(val_texts, val_labels, tokenizer, max_length=max_length)
test_ds = HFTextDataset(test_texts, test_labels, tokenizer, max_length=max_length)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Модель
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Используем ранее подготовленный base_kwargs для основного обучения
args = TrainingArguments(**base_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print(
    "Rebuilt trainer with best checkpoint:",
    checkpoint,
    "on device:",
    next(trainer.model.parameters()).device,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.260400
400,0.181400


distilbert-base-uncased: F1=0.5864


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.285800
400,0.201200


...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.308700
400,0.208900


roberta-base: F1=0.5670

Best checkpoint: distilbert-base-uncased (F1=0.5864)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rebuilt trainer with best checkpoint: distilbert-base-uncased on device: cuda:0


качество сосопоставимое , будем использовать distilbert-base-uncased

In [ ]:
# # Обучение
# train_result = trainer.train()
# print(train_result)

# # Сохраним лучшую модель и токенизатор
# best_dir = "./civil_comments_distilbert/best"
# trainer.save_model(best_dir)
# tokenizer.save_pretrained(best_dir)

Step,Training Loss
1000,0.194000
2000,0.160400
3000,0.159900
4000,0.150600
5000,0.142300
6000,0.139800
7000,0.150800
8000,0.141000
9000,0.139400
10000,0.152300


TrainOutput(global_step=112805, training_loss=0.12683433781944212, metrics={'train_runtime': 3186.9644, 'train_samples_per_second': 566.33, 'train_steps_per_second': 35.396, 'total_flos': 9.275657129817835e+16, 'train_loss': 0.12683433781944212, 'epoch': 1.0})


('./civil_comments_distilbert/best/tokenizer_config.json',
 './civil_comments_distilbert/best/special_tokens_map.json',
 './civil_comments_distilbert/best/vocab.txt',
 './civil_comments_distilbert/best/added_tokens.json',
 './civil_comments_distilbert/best/tokenizer.json')

## Итоговая проверка качества на тестовом наборе


In [43]:
# Тестовая оценка: метрики и краткий отчёт
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

# Оценка на тесте
test_metrics = trainer.evaluate(eval_dataset=test_ds)

# Детальные предсказания
preds_out = trainer.predict(test_ds)
logits = preds_out.predictions  # [N, 2]
labels = preds_out.label_ids    # [N]
preds = logits.argmax(axis=1)

f1 = float(f1_score(labels, preds))
precision = float(precision_score(labels, preds, zero_division=0))
recall = float(recall_score(labels, preds, zero_division=0))

In [44]:
print(f"F1:  {f1:.4f}")
print(f"Prec:{precision:.4f}")
print(f"Recall:{recall:.4f}")
print("eval dict:", test_metrics)

F1:  0.7026
Prec:0.7746
Recall:0.6429
eval dict: {'eval_loss': 0.11346084624528885, 'eval_f1': 0.7026419336706015, 'eval_precision': 0.774593338497289, 'eval_recall': 0.6429214350006429, 'eval_runtime': 63.4537, 'eval_samples_per_second': 1533.717, 'eval_steps_per_second': 47.94, 'epoch': 1.0}


In [45]:
# Отчёт по качеству: матрица ошибок и classification report
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(labels, preds)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(labels, preds, digits=4))

Confusion matrix:
 [[88088  1455]
 [ 2777  5000]]

Classification report:
               precision    recall  f1-score   support

           0     0.9694    0.9838    0.9765     89543
           1     0.7746    0.6429    0.7026      7777

    accuracy                         0.9565     97320
   macro avg     0.8720    0.8133    0.8396     97320
weighted avg     0.9539    0.9565    0.9547     97320



TN (истинно-нетоксичны): 88 088
FP (ложно помечены как токсичные): 1 455
FN (пропущенные токсичные): 2 777
TP (верно найденные токсичные): 5 000